In [74]:
import os
import warnings
warnings.filterwarnings("ignore")
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit, cross_validate, RandomizedSearchCV, GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, make_scorer
from sklearn.base import BaseEstimator, TransformerMixin
try: from lightgbm import LGBMRegressor
except Exception: LGBMRegressor = None
try:
    from xgboost import XGBRegressor
except ImportError:
    XGBRegressor = None

sns.set(style="whitegrid")
np.random.seed(42)

In [75]:
sns.set(style="whitegrid")
np.random.seed(42)

folder_tag="_lag" # modify path if lag features are used / planned to be used when lag features are not used


model_plot_path=f"artifacts/model_plots{folder_tag}/"
# model_train_path=f"artifacts/model_train_data{folder_tag}/"
model_path=f"artifacts/models{folder_tag}/"
model_results_path=f"artifacts/model_results{folder_tag}/"

# Create folders for artifacts & models
os.makedirs(model_plot_path, exist_ok=True)
# os.makedirs(model_train_path, exist_ok=True)
os.makedirs(model_path, exist_ok=True)
os.makedirs(model_results_path, exist_ok=True)

In [76]:
INPUT_CSV ="artifacts/data/clean_data.csv"  # change as needed

# Target variable name (ensure matches csv)
TARGET = "Average_Price"

# Which models to run (strings): "Linear", "RandomForest", "XGBoost", "LightGBM", "ARIMA"
RUN_MODELS = ["Linear", "RandomForest", "XGBoost", "LightGBM", "ARIMA"]

# TimeSeriesSplit folds
N_SPLITS = 5

# CV scoring metrics list (modify as needed)
# Will be used for final reporting. Keep names consistent with regression_metrics below.
METRIC_NAMES = ["RMSE", "MSE", "MAE", "MAPE", "R2"]

# Randomized / Grid search settings
RANDOM_SEARCH_ITER = 20
GRID_SEARCH_SMALL = True  # if True, run smaller grids to save time

# Use n_jobs=1 to avoid multiprocessing issues in constrained environments
# N_JOBS = 1

In [77]:
import joblib
col_categories = joblib.load(f"col_categories.joblib")

In [78]:
engineer_feature_cols = [
    "Average_Price", "Supply_Volume",
    # --- Climate features ---
    "Kathmandu_Rainfall_MM",
    "Hilly_Temperature", "Hilly_Precipitation", "Hilly_Rainfall_MM",
    "Sarlahi_Temperature", "Sarlahi_Precipitation", "Sarlahi_Rainfall_MM"
]

remove_columns = ['imported_tomato_price','Kathmandu_Wind_Speed', 'Kathmandu_Temperature', 
                  'Kathmandu_Precipitation','Dhading_Wind_Speed','Sarlahi_Wind_Speed', 'Sarlahi_Air_Pressure',
                  'Kavre_Air_Pressure','Kavre_Wind_Speed', 'Kathmandu_Air_Pressure',
                  "Dhading_Temperature", "Dhading_Air_Pressure","Dhading_Precipitation","Hilly_Precipitation", "Dhading_Rainfall_MM", "Kavre_Temperature", "Kavre_Precipitation", "Kavre_Rainfall_MM",
                  "Sarlahi_Precipitation"
                  ]

# Define per-column lag and roll configurations





In [79]:
def regression_metrics(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    mask = y_true != 0
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100 if mask.sum() > 0 else np.nan
    r2 = r2_score(y_true, y_pred)
    return {"RMSE": rmse, "MSE": mse, "MAE": mae, "MAPE": mape, "R2": r2}

sk_rmse = make_scorer(lambda y, yhat: -np.sqrt(mean_squared_error(y, yhat)))
sk_mse = make_scorer(lambda y, yhat: -mean_squared_error(y, yhat))
sk_mae = make_scorer(lambda y, yhat: -mean_absolute_error(y, yhat))
def sk_mape(y, yhat):
    mask = y != 0
    return -np.mean(np.abs((y[mask] - yhat[mask]) / y[mask])) * 100 if mask.sum() > 0 else 0
sk_mape_scorer = make_scorer(sk_mape)

SCORING = {"neg_rmse": sk_rmse, "neg_mse": sk_mse, "neg_mae": sk_mae, "neg_mape": sk_mape_scorer, "r2": "r2"}


In [80]:
df = pd.read_csv(INPUT_CSV, parse_dates=["Date"], infer_datetime_format=True)
df = df.dropna(subset=[TARGET]).reset_index(drop=True)
if "Date" in df.columns:
    df = df.sort_values("Date").reset_index(drop=True)

bool_cols = df.select_dtypes(include=["bool"]).columns
df[bool_cols] = df[bool_cols].astype(int)

cols_all = [c for c in df.columns if c not in [TARGET, "Date"]]
numeric_cols = df[cols_all].select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in cols_all if c not in numeric_cols]


In [81]:
df.head()

,Date,Average_Price,Supply_Volume,USD_TO_NPR,Diesel,is_festival,Season_Autumn,Season_Monsoon,Season_Spring,Season_Winter,...,Sarlahi_Rainfall_MM,Hilly_Temperature,Hilly_Precipitation,Hilly_Rainfall_MM,day,month,day_of_week,is_weekend,month_sin,month_cos
0,2022-01-01,115.00,8000.0,119.24,119.0,0,0,0,0,1,...,0.0,10.193583,0.0,0.0,1,1,5,1,0.5,0.866025
1,2022-01-02,115.00,8000.0,119.24,119.0,0,0,0,0,1,...,0.0,10.681083,0.0,0.0,2,1,6,1,0.5,0.866025
2,2022-01-03,95.00,22375.0,119.24,119.0,0,0,0,0,1,...,0.0,11.019625,0.0,0.0,3,1,0,0,0.5,0.866025
3,2022-01-04,96.67,8000.0,119.12,119.0,0,0,0,0,1,...,0.0,10.984209,0.0,0.0,4,1,1,0,0.5,0.866025
4,2022-01-05,86.67,32500.0,119.59,119.0,0,0,0,0,1,...,0.0,11.434209,0.0,0.0,5,1,2,0,0.5,0.866025


In [82]:
class LagRollTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, lag_config=None, roll_config=None, use_std=False):
        self.lag_config = lag_config or {}
        self.roll_config = roll_config or {}
        self.use_std = use_std
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X_new = X.copy()
        for col, lags in self.lag_config.items():
            for lag in lags:
                X_new[f"{col}_lag{lag}"] = X_new[col].shift(lag)
        for col, windows in self.roll_config.items():
            for w in windows:
                X_new[f"{col}_rollmean_{w}"] = X_new[col].rolling(window=w).mean()
                if self.use_std:
                    X_new[f"{col}_rollstd_{w}"] = X_new[col].rolling(window=w).std()
        # REMOVE or comment this line:
        # X_new = X_new.dropna().reset_index(drop=True)
        return X_new


In [83]:
def save_plot(fig, name):
    filepath = os.path.join(model_plot_path, f"{name}.png")
    fig.savefig(filepath, bbox_inches="tight", dpi=200)
    plt.close(fig)
    print("Saved:", filepath)


In [84]:
# --------- 3. Load data ---------
df = pd.read_csv(INPUT_CSV, parse_dates=["Date"], infer_datetime_format=True)
print("Loaded:", INPUT_CSV, " shape:", df.shape)

# Quick drop rows with missing target
df = df.dropna(subset=[TARGET]).reset_index(drop=True)

bool_cols = df.select_dtypes(include=["bool"]).columns
df[bool_cols] = df[bool_cols].astype(int)

# Separate features & target, keep Date for potential time-splits/plots
if "Date" in df.columns:
    df = df.sort_values("Date").reset_index(drop=True)


Loaded: artifacts/data/clean_data.csv  shape: (1389, 24)


In [85]:
df_no_lag = df.copy()


In [86]:
cols_all = [c for c in df.columns if c not in [TARGET]]

# # Exclude direct leakage: if any column equal to TARGET or 'imported_tomato_price' recorded same time, you can adjust here
# if "imported_tomato_price" in cols_all:
#     # drop from modeling if same-day leak for forecasting; user insisted earlier that it's leakage for forecasting
#     cols_all.remove("imported_tomato_price")
#     print("Removed 'imported_tomato_price' from features (contemporaneous leak).")

# Drop Date from features
if "Date" in cols_all:
    cols_all.remove("Date")

In [87]:
# Identify numeric and categorical automatically
numeric_cols = df[cols_all].select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in cols_all if c not in numeric_cols]

print("NUMERIC cols:", numeric_cols)
print("CATEGORICAL cols:", cat_cols)


NUMERIC cols: ['Supply_Volume', 'USD_TO_NPR', 'Diesel', 'is_festival', 'Season_Autumn', 'Season_Monsoon', 'Season_Spring', 'Season_Winter', 'Inflation', 'Kathmandu_Rainfall_MM', 'Sarlahi_Temperature', 'Sarlahi_Precipitation', 'Sarlahi_Rainfall_MM', 'Hilly_Temperature', 'Hilly_Precipitation', 'Hilly_Rainfall_MM', 'day', 'month', 'day_of_week', 'is_weekend', 'month_sin', 'month_cos']
CATEGORICAL cols: []


In [88]:
# # For no-lag experiment, optionally drop lag columns (heuristic: columns with '_lag' or '_roll' in name)
# def filter_no_lag_columns(cols):
#     return [c for c in cols if (("_lag" not in c) and ("_roll" not in c) and ("lag_" not in c) and ("roll" not in c))]


In [89]:
# Create feature sets
FEATURES_FULL = cols_all                                    # all features present in CSV
# FEATURES_NO_LAG = filter_no_lag_columns(FEATURES_FULL)      # remove lag/roll columns for no-lag model
FEATURES_FULL = [c for c in cols_all if c != TARGET]  # exclude target
print("FEATURES_FULL count:", len(FEATURES_FULL))
# print("FEATURES_NO_LAG count:", len(FEATURES_NO_LAG))


FEATURES_FULL count: 22


In [ ]:
# Numeric pipeline: impute (ffill is already used earlier but we still include a safe imputer) + scaling
def build_and_save_preprocessor(df, cols_all, save_path="artifacts/preprocessor/preprocessor.pkl"):
    # Auto-detect numeric & categorical
    numeric_cols = df[cols_all].select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = [c for c in cols_all if c not in numeric_cols]

    print(f"NUMERIC: {len(numeric_cols)} | CATEGORICAL: {len(cat_cols)}")

    # Numeric pipeline
    num_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    # Categorical pipeline
    cat_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    # ColumnTransformer
    preprocessor = ColumnTransformer(transformers=[
        ("num", num_pipeline, numeric_cols),
        ("cat", cat_pipeline, cat_cols)
    ], remainder="drop", sparse_threshold=0)

    # Fit it once so it knows categories and feature order
    preprocessor.fit(df[cols_all])

    # Save both preprocessor and schema
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    joblib.dump(preprocessor, save_path)
    joblib.dump({
        "numeric_cols": numeric_cols,
        "cat_cols": cat_cols,
        "all_cols": cols_all
    }, os.path.join(os.path.dirname(save_path), "feature_schema.pkl"))

    print(f"Preprocessor saved to {save_path}")
    return preprocessor

# Helper to build a full sklearn Pipeline for a given estimator
preprocessor=build_and_save_preprocessor(df,cols_all)

def make_pipeline(estimator):
    return Pipeline([
        ("preproc", preprocessor),
        ("est", estimator)
    ])


def make_pipeline_with_lag(estimator):
    
custom_roll_config = {
"Average_Price": [1,7],
"Supply_Volume": [3],
# "imported_tomato_price": [7]
}

custom_lag_config = {
    # --- Target and market ---
    "Average_Price": [1, 3, 7],
    "Supply_Volume": [1, 7, 30],

    # --- Climate lags ---
    # Sarlahi (production region)
    "Sarlahi_Temperature": [1, 3, 7],
    # "Sarlahi_Precipitation": [1, 3, 7],
    "Sarlahi_Rainfall_MM": [1, 3, 7],

    # Hilly (Kavre+Dhading merged region)
    "Hilly_Temperature": [1, 3, 7],
    # "Hilly_Precipitation": [1, 3, 7],
    "Hilly_Rainfall_MM": [1, 3, 7],

    # Kathmandu (logistics/weather impact)
    "Kathmandu_Rainfall_MM": [1, 2]
}

standard_deviation_lag = False        
   
# ----------- 2. Apply LagRollTransformer -----------
lag_transformer = LagRollTransformer(
    lag_config=custom_lag_config,
    roll_config=custom_roll_config,
    use_std=standard_deviation_lag
)

X_lagged = lag_transformer.fit_transform(df[FEATURES_FULL])

# ----------- 3. Drop initial rows with NaNs -----------
max_lag = max([max(v) for v in custom_lag_config.values()])
max_roll = max([max(v) for v in custom_roll_config.values()])
drop_rows = max(max_lag, max_roll)

X_lagged = X_lagged.iloc[drop_rows:].reset_index(drop=True)
y_aligned = df[TARGET].iloc[drop_rows:].reset_index(drop=True)

print("After lag/roll -> X shape:", X_lagged.shape, "y shape:", y_aligned.shape)

# ----------- 4. Build preprocessor on lagged features -----------
preprocessor = build_and_save_preprocessor(
    pd.concat([X_lagged, y_aligned], axis=1),
    X_lagged.columns.tolist(),
    save_path=f"artifacts/preprocessor/preprocessor_lag.pkl"
)

NUMERIC: 22 | CATEGORICAL: 0
Preprocessor saved to artifacts/preprocessor/preprocessor.pkl


In [91]:
# ----------- 5. Define pipeline function with lag support -----------
def make_pipeline_with_lag(estimator):
    return Pipeline([
        ("preproc", preprocessor),
        ("est", estimator)
    ])

In [92]:
X_final = preprocessor.transform(df[FEATURES_FULL])
feature_schema = list(FEATURES_FULL)

with open(f"feature_schema{folder_tag}.json", "w") as f:
    json.dump(feature_schema, f)


In [93]:
# ----------- 6. Create model pipelines -----------
models = {}
param_grids = {}

models["Linear"] = make_pipeline_with_lag(LinearRegression())
param_grids["Linear"] = {}

models["RandomForest"] = make_pipeline_with_lag(RandomForestRegressor(random_state=42))
param_grids["RandomForest"] = {
    "est__n_estimators": [100, 200],
    "est__max_depth": [5, 10, None],
    "est__min_samples_leaf": [1, 2, 4]
}

if XGBRegressor is not None:
    models["XGBoost"] = make_pipeline_with_lag(XGBRegressor(objective="reg:squarederror", random_state=42))
    param_grids["XGBoost"] = {
        "est__n_estimators": [100, 200],
        "est__max_depth": [3, 5],
        "est__learning_rate": [0.01, 0.05, 0.1]
    }

if LGBMRegressor is not None:
    models["LightGBM"] = make_pipeline_with_lag(LGBMRegressor(random_state=42))
    param_grids["LightGBM"] = {
        "est__n_estimators": [100, 200],
        "est__num_leaves": [31, 64],
        "est__learning_rate": [0.01, 0.05]
    }

# ----------- 7. Train-test split (chronological holdout) -----------
holdout_frac = 0.2
n_holdout = int(len(X_lagged) * holdout_frac)
train_idx = slice(0, len(X_lagged) - n_holdout)
test_idx = slice(len(X_lagged) - n_holdout, len(X_lagged))

X_train_final = X_lagged.iloc[train_idx].reset_index(drop=True)
y_train_final = y_aligned.iloc[train_idx].reset_index(drop=True)
X_test_final  = X_lagged.iloc[test_idx].reset_index(drop=True)
y_test_final  = y_aligned.iloc[test_idx].reset_index(drop=True)

print("Train shape:", X_train_final.shape, "Test shape:", X_test_final.shape)
# Ensure no NA in X due to feature selection; imputer in pipeline handles remaining NAs.
print("X shape:", X.shape, " y shape:", y.shape)

tscv = TimeSeriesSplit(n_splits=N_SPLITS)

# Storage for results
cv_summary = []
best_models = {}

# Iterate models for training and hyperparameter tuning
for name, pipeline in models.items():
    if name not in RUN_MODELS:
        continue
    print("\n" + "="*40)
    print("Training model:", name)
    print("="*40)

    grid = param_grids.get(name, None)

    # If no grid (e.g., Linear), just cross-validate without search
    if not grid:
        print("No hyperparameter grid for", name, " — running cross_validate with TimeSeriesSplit.")
        cv_res = cross_validate(
        pipeline, X, y,
        cv=tscv,
        scoring=SCORING,
        return_train_score=True
        )

        # Compute average metrics from cv_res (note neg scorers are negative)
        # Convert negative scorers to positive metrics
        results = {
            "Model": name,
            "RMSE_mean": -np.mean(cv_res["test_neg_rmse"]) if "test_neg_rmse" in cv_res else np.nan,
            "MSE_mean": -np.mean(cv_res["test_neg_mse"]) if "test_neg_mse" in cv_res else np.nan,
            "MAE_mean": -np.mean(cv_res["test_neg_mae"]) if "test_neg_mae" in cv_res else np.nan,
            "MAPE_mean": -np.mean(cv_res["test_neg_mape"]) if "test_neg_mape" in cv_res else np.nan,
            "R2_mean": np.mean(cv_res["test_r2"]) if "test_r2" in cv_res else np.nan
        }
        cv_summary.append(results)

        # Fit on full training portion (we will treat the last 20% as test later)
        fitted = pipeline.fit(X, y)
        best_models[name] = fitted
        # Save fitted model
        joblib.dump(fitted, f"{model_path}{name}_best.joblib")
        print("Saved model:", f"{model_path}{name}_best.joblib")
        continue

    # If grid provided -> run RandomizedSearch then refine with GridSearch (optional)
    # Randomized Search (broad)
    print("Running RandomizedSearchCV (broad) for", name)
    rnd = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=grid,
        n_iter=RANDOM_SEARCH_ITER,
        cv=tscv,
        scoring="neg_mean_squared_error",   # use neg MSE for search ranking
        random_state=42,
        # n_jobs=N_JOBS,
        verbose=1
    )
    rnd.fit(X, y)
    print("RandomSearch best params:", rnd.best_params_, " best_score:", rnd.best_score_)

    # Optionally run a smaller GridSearch around the best params (if desired)
    # We will attempt a small grid: replace the param with the found best if not present
    if GRID_SEARCH_SMALL:
        # build a small grid based on rnd.best_params_ if possible
        small_grid = {}
        for k, v in grid.items():
            if k in rnd.best_params_:
                # if the best param is inside the list of grid values, pick neighbors or keep list small
                small_choices = grid[k]
                small_grid[k] = small_choices if len(small_choices) <= 3 else small_choices[:3]
            else:
                small_grid[k] = grid[k] if isinstance(grid[k], list) else [grid[k]]
        print("Running GridSearchCV (refined) for", name)
        gscv = GridSearchCV(
            estimator=pipeline,
            param_grid=small_grid,
            cv=tscv,
            scoring="neg_mean_squared_error",
            # n_jobs=N_JOBS,
            verbose=1
        )
        gscv.fit(X, y)
        best_est = gscv.best_estimator_
        best_params = gscv.best_params_
        best_score = gscv.best_score_
        print("GridSearch best params:", best_params, "best_score:", best_score)
    else:
        best_est = rnd.best_estimator_
        best_params = rnd.best_params_
        best_score = rnd.best_score_

    # Store best estimator
    best_models[name] = best_est
    # Save model to disk
    joblib.dump(best_est, f"{model_path}{name}_best.joblib")
    print("Saved best model:", f"{model_path}{name}_best.joblib")

    # Cross-validate the best estimator to get metrics per-fold
    cv_res = cross_validate(best_est, X, y, cv=tscv, scoring=SCORING, return_train_score=False)
    results = {
        "Model": name,
        "RMSE_mean": -np.mean(cv_res["test_neg_rmse"]) if "test_neg_rmse" in cv_res else np.nan,
        "RMSE_std" : np.std([-np.mean(cv_res["test_neg_rmse"])]),
        "MSE_mean": -np.mean(cv_res["test_neg_mse"]) if "test_neg_mse" in cv_res else np.nan,
        "MAE_mean": -np.mean(cv_res["test_neg_mae"]) if "test_neg_mae" in cv_res else np.nan,
        "MAPE_mean": -np.mean(cv_res["test_neg_mape"]) if "test_neg_mape" in cv_res else np.nan,
        "R2_mean": np.mean(cv_res["test_r2"]) if "test_r2" in cv_res else np.nan
    }
    cv_summary.append(results)


NameError: name 'X_lagged' is not defined

In [ ]:
# --- Train vs CV comparison (per model, per metric) ---

records = []

# rerun cross_validate to capture both train & test per fold
for name, pipeline in models.items():
    if name not in best_models:
        continue
    print(f"Evaluating train vs CV metrics for {name}")
    cv_res = cross_validate(
        pipeline,
        X, y,
        cv=tscv,
        scoring={
            "rmse": sk_rmse,
            "mse": sk_mse,
            "mae": sk_mae,
            "mape": sk_mape_scorer,
            "r2": "r2"
        },
        return_train_score=True
    )
    
    for metric_key in ["rmse", "mse", "mae", "mape", "r2"]:
        train_metric = cv_res.get(f"train_{metric_key}", None)
        test_metric = cv_res.get(f"test_{metric_key}", None)
        if train_metric is None or test_metric is None:
            continue

        # reverse neg sign for errors
        if "neg" in metric_key:
            train_metric = -train_metric
            test_metric = -test_metric

        for fold, (tr, ts) in enumerate(zip(train_metric, test_metric), 1):
            records.append({
                "Model": name,
                "Metric": metric_key.upper(),
                "Fold": fold,
                "Train": tr,
                "CV": ts
            })

df_comp = pd.DataFrame(records)
print(df_comp.head())


Evaluating train vs CV metrics for Linear
Evaluating train vs CV metrics for RandomForest
Evaluating train vs CV metrics for XGBoost
Evaluating train vs CV metrics for LightGBM
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000169 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2418
[LightGBM] [Info] Number of data points in the train set: 229, number of used features: 50
[LightGBM] [Info] Start training from score 72.776419
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further 

In [ ]:
for metric in df_comp["Metric"].unique():
    fig, ax = plt.subplots(figsize=(8, 4))
    dfp = df_comp[df_comp["Metric"] == metric]
    df_melt = dfp.melt(
        id_vars=["Model", "Fold", "Metric"],
        value_vars=["Train", "CV"],
        var_name="Set",
        value_name="Score"
    )
    sns.barplot(x="Model", y="Score", hue="Set", data=df_melt, ax=ax)
    ax.set_title(f"Train vs CV — {metric}")
    ax.set_ylabel(metric)
    plt.xticks(rotation=45)
    save_plot(fig, f"train_vs_cv_{metric.lower()}_by_model")


Saved: artifacts/model_plots_lag/train_vs_cv_rmse_by_model.png
Saved: artifacts/model_plots_lag/train_vs_cv_mse_by_model.png
Saved: artifacts/model_plots_lag/train_vs_cv_mae_by_model.png
Saved: artifacts/model_plots_lag/train_vs_cv_mape_by_model.png
Saved: artifacts/model_plots_lag/train_vs_cv_r2_by_model.png


In [ ]:
# We will use the last 20% of chronological data as test set (time-based holdout)
holdout_frac = 0.2
n_holdout = int(len(X) * holdout_frac)
if n_holdout < 1:
    raise RuntimeError("Dataset too small for holdout fraction.")

train_idx = slice(0, len(X) - n_holdout)
test_idx = slice(len(X) - n_holdout, len(X))

X_train_final = X.iloc[train_idx].reset_index(drop=True)
y_train_final = y.iloc[train_idx].reset_index(drop=True)
X_test_final  = X.iloc[test_idx].reset_index(drop=True)
y_test_final  = y.iloc[test_idx].reset_index(drop=True)

In [ ]:
# Concatenate X and y for easier saving
train_save = pd.concat([X_train_final, y_train_final], axis=1)
test_save  = pd.concat([X_test_final, y_test_final], axis=1)

# Save CSVs
train_save.to_csv(f"artifacts/data/train_data{folder_tag}.csv", index=False)
test_save.to_csv(f"artifacts/data/test_data{folder_tag}.csv", index=False)

In [ ]:
print("\nFinal holdout sizes -> Train:", X_train_final.shape, " Test:", X_test_final.shape)


Final holdout sizes -> Train: (1088, 51)  Test: (271, 51)


In [ ]:
eval_records = []

for name, model in best_models.items():
    print("\nEvaluating on holdout:", name)
    
    # Ensure model is fitted
    if hasattr(model, "fit") and (not hasattr(model, "predict") or getattr(model, "steps", None) is None):
        model.fit(X_train_final, y_train_final)
    
    # Predict
    y_pred = model.predict(X_test_final)
    
    # Compute metrics
    mse = mean_squared_error(y_test_final, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test_final, y_pred)
    r2 = r2_score(y_test_final, y_pred)
    
    eval_records.append({
        "Model": name,
        "RMSE": rmse,
        "MSE": mse,
        "MAE": mae,
        "R2": r2
    })
    
    # --- Plot ---
    fig, ax = plt.subplots(figsize=(10, 3))
    if "Date" in df.columns:
        test_dates = df["Date"].iloc[test_idx].reset_index(drop=True)
        ax.plot(test_dates, y_test_final, label="Actual")
        ax.plot(test_dates, y_pred, linestyle="--", label="Predicted")
        fig.autofmt_xdate()
    else:
        ax.plot(y_test_final.index, y_test_final, label="Actual")
        ax.plot(y_test_final.index, y_pred, linestyle="--", label="Predicted")
    
    ax.set_title(f"{name} — Actual vs Predicted (Holdout)")
    ax.legend()
    plt.tight_layout()
    save_plot(fig, f"actual_vs_pred_{name}")

# Save results
eval_df = pd.DataFrame(eval_records).sort_values("RMSE")
eval_df.to_csv(f"{model_results_path}holdout_performance.csv", index=False)
display(eval_df.round(4))


Evaluating on holdout: Linear
Saved: artifacts/model_plots_lag/actual_vs_pred_Linear.png

Evaluating on holdout: RandomForest
Saved: artifacts/model_plots_lag/actual_vs_pred_RandomForest.png

Evaluating on holdout: XGBoost
Saved: artifacts/model_plots_lag/actual_vs_pred_XGBoost.png

Evaluating on holdout: LightGBM
Saved: artifacts/model_plots_lag/actual_vs_pred_LightGBM.png


,Model,RMSE,MSE,MAE,R2
0,Linear,0.0000,0.0000,0.0000,1.0000
1,RandomForest,0.1679,0.0282,0.0558,0.9999
2,XGBoost,0.1903,0.0362,0.1067,0.9999
3,LightGBM,0.3350,0.1122,0.1066,0.9996


In [ ]:
cv_df = pd.DataFrame(cv_summary)
cv_df.to_csv(f"{model_results_path}cv_summary.csv", index=False)
print(f"Saved CV summary to {model_results_path}cv_summary.csv")
display(cv_df.round(4))

# Plot CV RMSE comparison
if "RMSE_mean" in cv_df.columns:
    fig, ax = plt.subplots(figsize=(8,4))
    sns.barplot(x="Model", y="RMSE_mean", data=cv_df, dodge=False, ax=ax)
    ax.set_title("CV: Mean RMSE by Model")
    save_plot(fig, "cv_rmse_by_model")

# Plot holdout RMSE
fig, ax = plt.subplots(figsize=(8,4))
sns.barplot(x="Model", y="RMSE", data=eval_df.rename(columns={"RMSE":"RMSE"}), dodge=False, ax=ax)
ax.set_title("Holdout: RMSE by Model")
save_plot(fig, "holdout_rmse_by_model")




Saved CV summary to artifacts/model_results_lag/cv_summary.csv


,Model,RMSE_mean,MSE_mean,MAE_mean,MAPE_mean,R2_mean,RMSE_std
0,Linear,0.0000,0.0000,0.0000,0.0000,1.0000,NaN
1,RandomForest,1.9932,8.4746,0.6722,1.1996,0.9856,0.0
2,XGBoost,1.5447,5.1707,0.4865,0.8467,0.9907,0.0
3,LightGBM,4.0319,27.8973,2.0484,4.0593,0.9432,0.0


Saved: artifacts/model_plots_lag/cv_rmse_by_model.png
Saved: artifacts/model_plots_lag/holdout_rmse_by_model.png


In [ ]:
# --------- 10. Feature importance for tree models (Permutation or built-in) ---------
# If RandomForest present, extract feature importances (via fitted pipeline)
if "RandomForest" in best_models:
    rf = best_models["RandomForest"]
    try:
        # Get feature names after preprocessing
        pre = rf.named_steps["preproc"]
        # numeric names and onehot names
        num_names = numeric_cols
        # get onehot feature names if any categories
        try:
            ohe = pre.named_transformers_["cat"].named_steps["onehot"]
            ohe_names = list(ohe.get_feature_names_out(cat_cols))
        except Exception:
            ohe_names = []
        feat_names = num_names + ohe_names

        # Extract underlying RandomForest
        rf_est = rf.named_steps["est"]
        importances = getattr(rf_est, "feature_importances_", None)
        if importances is not None:
            imp_df = pd.DataFrame({"feature": feat_names, "importance": importances})
            imp_df = imp_df.sort_values("importance", ascending=False).head(30)
            imp_df.to_csv(f"{model_results_path}rf_feature_importances.csv", index=False)

            fig, ax = plt.subplots(figsize=(8,6))
            sns.barplot(x="importance", y="feature", data=imp_df, ax=ax)
            ax.set_title("RandomForest: Top feature importances")
            save_plot(fig, "rf_top_feature_importances")
    except Exception as e:
        print("Failed to extract RF feature importances:", e)

Saved: artifacts/model_plots_lag/rf_top_feature_importances.png


In [ ]:
if "ARIMA" in RUN_MODELS and ARIMA is not None:
    print("\nRunning simple ARIMA baseline on target (univariate) ...")
    try:
        series = df.set_index("Date")[TARGET].asfreq("D").interpolate()
        arima_order = (1,1,1)   # simple baseline
        arima_model = ARIMA(series.iloc[:-n_holdout], order=arima_order).fit()
        arima_fore = arima_model.forecast(steps=n_holdout)
        mets = regression_metrics(series.iloc[-n_holdout:].values, arima_fore)
        mets["Model"] = "ARIMA"
        # append to eval records and save plot
        eval_df = pd.concat([eval_df, pd.DataFrame([mets])], ignore_index=True)
        # Plot
        fig, ax = plt.subplots(figsize=(10,3))
        ax.plot(series.index[-n_holdout:], series.iloc[-n_holdout:], label="Actual")
        ax.plot(series.index[-n_holdout:], arima_fore, linestyle="--", label="ARIMA_Forecast")
        ax.set_title("ARIMA: Actual vs Forecast (holdout)")
        ax.legend()
        save_plot(fig, "arima_holdout")
        # Save ARIMA model
        joblib.dump(arima_model, "artifacts/models/arima_baseline.joblib")
        print("Saved ARIMA baseline model.")
    except Exception as e:
        print("ARIMA failed:", e)


Running simple ARIMA baseline on target (univariate) ...
Saved: artifacts/model_plots_lag/arima_holdout.png
Saved ARIMA baseline model.


In [ ]:
manifest = {
    "timestamp": datetime.utcnow().isoformat(),
    "input_csv": INPUT_CSV,
    "target": TARGET,
    "features_used": FEATURES_TO_USE,
    "numeric_cols": numeric_cols,
    "cat_cols": cat_cols,
    "models_trained": list(best_models.keys()),
    "holdout_rows": int(n_holdout)
}
with open(f"{model_results_path}manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)
print(f"Saved manifest -> {model_results_path}manifest.json")

print("\nNotebook 2 complete. Artifacts saved under artifacts/model_plots, artifacts/models, artifacts/model_results.")

Saved manifest -> artifacts/model_results_lag/manifest.json

Notebook 2 complete. Artifacts saved under artifacts/model_plots, artifacts/models, artifacts/model_results.


In [ ]:
print(col_categories)

In [ ]:
import joblib
joblib.dump(col_categories, "col_categories_lag.joblib")